In [16]:
import pandas as pd

In [17]:
import os

mapping_file = pd.ExcelFile(os.path.join( 'metadata', 'vietnames_to_fiin.xlsx'))

In [18]:
df_bs = mapping_file.parse('Standard BS')[['Universal_caption', 'universal_code']]
df_is = mapping_file.parse('Standard IS')[['Universal_caption', 'universal_code']]
df_cf = mapping_file.parse('Standard CF')[['Universal_caption', 'universal_code']]

In [19]:
df_bs = df_bs.rename(columns={'Universal_caption': 'category_name', 'universal_code': 'category_code'})
df_is = df_is.rename(columns={'Universal_caption': 'category_name', 'universal_code': 'category_code'})
df_cf = df_cf.rename(columns={'Universal_caption': 'category_name', 'universal_code': 'category_code'})

df_map_fs = pd.concat([df_bs, df_is, df_cf], ignore_index=True).dropna(subset=['category_code'])

df_map_fs.to_csv(os.path.join('metadata', 'map_category_code.csv'), index=False)


In [20]:
df_map_tm_bank = mapping_file.parse('fiin_TM_bank')
df_map_tm_sec = mapping_file.parse('fiin_TM_sec')
df_map_tm_corp = mapping_file.parse('fiin_TM_corp')


df_map_tm_bank['category_code'] = df_map_tm_bank['category_code'].apply(lambda x: "Bank_" + x)
df_map_tm_sec['category_code'] = df_map_tm_sec['category_code'].apply(lambda x: "Sec_" + x)
df_map_tm_corp['category_code'] = df_map_tm_corp['category_code'].apply(lambda x: "Corp_" + x)


map_share_sec = ~df_map_tm_sec['share_code'].isna()
map_share_corp = ~df_map_tm_corp['share_code'].isna()

df_map_tm_sec.loc[map_share_sec, 'category_code'] = df_map_tm_sec.loc[map_share_sec, 'share_code']
df_map_tm_corp.loc[map_share_corp, 'category_code'] = df_map_tm_corp.loc[map_share_corp, 'share_code']


df_map_tm_bank = df_map_tm_bank[['category_code', 'vi_caption', 'en_caption', 'parent_code']]
df_map_tm_sec = df_map_tm_sec[['category_code', 'vi_caption', 'en_caption', 'parent_code']]
df_map_tm_corp = df_map_tm_corp[['category_code', 'vi_caption', 'en_caption', 'parent_code']]

df_map_tm_sec.dropna(subset=['en_caption'], inplace=True)
df_map_tm_bank.dropna(subset=['en_caption'], inplace=True)
df_map_tm_corp.dropna(subset=['en_caption'], inplace=True)

df_tm_map = pd.concat([df_map_tm_bank, df_map_tm_sec, df_map_tm_corp], ignore_index=True)
df_tm_map.drop_duplicates(subset=['category_code'], inplace=True)
df_tm_map['en_caption'] = df_tm_map['en_caption'].apply(lambda x: "(Explaination) " + x)


In [21]:
df_map_fs

,category_name,category_code
0,A. CURRENT ASSETS,BS_100
1,(Securities) Financial Assets,BS_101
2,Cash and cash equivalents,BS_110
3,Cash,BS_111
4,(Bank) Deposits at the Central Bank,BS_112
...,...,...
341,Net cash flow from financing activities,CF_090
342,Net cash flow during the period,CF_100
343,Cash and cash equivalents at the beginning of ...,CF_110
344,Effects of exchange rate changes on foreign cu...,CF_120


In [22]:
df_tm_map.rename(columns={'en_caption': 'category_name'}, inplace=True)
df_tm_map = df_tm_map[['category_code', 'category_name']]
df_map = pd.concat([df_map_fs, df_tm_map], ignore_index=True)

df_map.to_csv(os.path.join('metadata', 'map_category_code.csv'), index=False)


In [23]:
df = pd.read_parquet('../../data/financial_statement_v3.parquet')

In [24]:
df

,data,stock_code,quarter,year,date_added,segment,category_code
0,3253.384000,BID,0.0,2010.0,2010-12-30,bank,BS_111
1,3628.604000,BID,0.0,2011.0,2011-12-30,bank,BS_111
2,3295.068000,BID,0.0,2012.0,2012-12-30,bank,BS_111
3,3862.664000,BID,0.0,2013.0,2013-12-30,bank,BS_111
4,5393.484000,BID,0.0,2014.0,2014-12-30,bank,BS_111
...,...,...,...,...,...,...,...
422007,2612.819726,"Utilities (Electricity, Water & Gas)",3.0,2025.0,2025-09-30,industry,IS_100
422008,2549.118934,"Utilities (Electricity, Water & Gas)",3.0,2025.0,2025-09-30,industry,IS_103
422009,63.700792,"Utilities (Electricity, Water & Gas)",3.0,2025.0,2025-09-30,industry,IS_104
422010,1067.000000,"Utilities (Electricity, Water & Gas)",3.0,2025.0,2025-09-30,industry,IS_130


In [ ]:
df = df.dropna(subset=['category_code'])

In [19]:
df.to_parquet('../../data/financial_statement_v3.parquet', index=False)

In [16]:
df_tm = pd.read_parquet('../../data/financial_statement_explaination_v3.parquet')

In [17]:
df = pd.concat([df, df_tm])

In [18]:
df

,data,stock_code,quarter,year,date_added,category_code
0,0.0,SSI,0.0,2010.0,2010-12-30,IS_003
1,0.0,SSI,0.0,2011.0,2011-12-30,IS_003
2,0.0,SSI,0.0,2012.0,2012-12-30,IS_003
3,0.0,SSI,0.0,2013.0,2013-12-30,IS_003
4,0.0,SSI,0.0,2014.0,2014-12-30,IS_003
...,...,...,...,...,...,...
218043,0.0,DGC,3.0,2024.0,2024-09-30,Corp_TM_157
218044,0.0,DGC,4.0,2024.0,2024-12-30,Corp_TM_157
218045,0.0,DGC,1.0,2025.0,2025-03-30,Corp_TM_157
218046,0.0,DGC,2.0,2025.0,2025-06-30,Corp_TM_157


In [5]:
df[df['category_code'] == 'IS_090']

,data,stock_code,quarter,year,date_added,universal_code_x,universal_code_y,category_code
720,0.000000,SSI,0.0,2010.0,2010-12-30,None,None,IS_090
721,0.000000,SSI,0.0,2011.0,2011-12-30,None,None,IS_090
722,0.000000,SSI,0.0,2012.0,2012-12-30,None,None,IS_090
723,0.000000,SSI,0.0,2013.0,2013-12-30,None,None,IS_090
724,-183.144865,SSI,0.0,2014.0,2014-12-30,None,None,IS_090
...,...,...,...,...,...,...,...,...
157085,-61.995634,DGC,3.0,2024.0,2024-09-30,None,None,IS_090
157086,-79.776846,DGC,4.0,2024.0,2024-12-30,None,None,IS_090
157087,-141.122729,DGC,1.0,2025.0,2025-03-30,None,None,IS_090
157088,-95.468665,DGC,2.0,2025.0,2025-06-30,None,None,IS_090


In [1]:
from vnstock import Vnstock
import pandas as pd
import numpy as np

stock = Vnstock().stock(symbol='ACB', source='VCI')


## 👋 Chào mừng bạn đến với Vnstock!

Cảm ơn bạn đã sử dụng package phân tích chứng khoán #1 tại Việt Nam

* Tài liệu: [Sổ tay hướng dẫn](https://vnstocks.com/docs/category/s%E1%BB%95-tay-h%C6%B0%E1%BB%9Bng-d%E1%BA%ABn)
* Cộng đồng: [Nhóm Facebook](https://www.facebook.com/groups/vnstock.official)

Khám phá các tính năng mới nhất và tham gia cộng đồng để nhận hỗ trợ.
                

Phiên bản Vnstock 3.4.2 đã có mặt, vui lòng cập nhật với câu lệnh : `pip install vnstock --upgrade`.
Lịch sử phiên bản: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban
Phiên bản hiện tại 3.2.4

Phiên bản Vnai 2.3.9 đã có mặt, vui lòng cập nhật với câu lệnh : `pip install vnai --upgrade`.
Lịch sử phiên bản: https://pypi.org/project/vnai/#history
Phiên bản hiện tại 2.0.4

In [2]:
non_bank_stock_code = ["HSG", "ELC", "VSC", "ACV", "REE", "SZC", "CSV", "PAN", "BSR", "SGP", "GMD", "ITD","FOX", "KDC", "SBT", "VGC", "HBC", "CTD", "DIG", "SCR", "KBC","MWG", "NHA", "VNM", "HPG", "VHM", "PNJ", "YEG", "FPT","MSN", "GAS", "VRE", "VJC", "VIC", "PLX", "SAB", "POW", "GVR", "BCM", "VPI", "DVM", "KDH", "HDC", "TCH", "CEO", "HUT", "NVL", "DBC", "SAF", "DHT", "VTP", "PVT", "FRT", "DGC", "DCM", "NKG", "CMG", "VGI", "PVC", "CAP", "DTD", "HLD", "L14", "L18", "LAS", "LHC", "NTP", "PLC", "PSD", "PVG", "PVS", "SLS", "TIG", "TMB", "TNG", "TVD", "VC3", "VCS", "DXG"]
bank_stock_code = ["BID", "EIB", "OCB", "CTG", "VCB", "ACB", "MBB", "HDB", "TPB", "VPB",  "STB", "TCB",  "SHB", "VIB", "CTG",  "ABB", "LPB", "NVB"]
securities_stock_code = ["MBS", "VND", "SSI", "VIX", "ORS"]

new = False
if new:
    bank_stock_code += ["EVF", "MSB", "NAB", "SGB", "VAB", "KLB", "BVB", "PGB", "NVB"]
    securities_stock_code += ["VCI", "TVS", "SHS", "DSE", "HCM", "VDS", "APG", "CTS", "AGR"]
    non_bank_stock_code += ["VCG", "LCG", "DPG", "CTI", "TIS", "PSH", "TLH", "STK", "TDG", "TVN", "DRI", "CNG", "SIP", "PVP", "VGS", "VHC", "IJC", "CII", "SJS", "NLG", "NT2", "LBM", "PC1", "HAH", "HAG", "KOS", "RAL", "PHR", "ILB", "AGG", "ASM", "CLL", "CRE", "D2D", "UIC", "TMS", "PPC", "PTB", "VOS", "VIP", "VTO", "NRC", "GEX", "HTN", "VLC", "GEE", "TCM", "PDR", "SCR", "HPX", "LDG", "AAA", "CTR", "SAM", "BWE", "SGT", "BMP", "ITC", "TTN", "VTE", "VHC", "MSH", "TTF", "ANV", "CMX", "IDI", "FMC", "BAF", "HT1", "PVD", "FIT", "ACL", "ABT", "AAM", "PET", "DGW", "DPM", "HDG", "IMP", "MAS"]


symbols = non_bank_stock_code + bank_stock_code + securities_stock_code 

In [3]:
dfs = []
for stock_code in symbols:
    stock = Vnstock().stock(symbol=stock_code, source='VCI')
    df = stock.quote.history(start='2015-01-01', end='2025-01-01', interval='1M')
    df['stock_code'] = stock_code
    dfs.append(df)
df_stock_price = pd.concat(dfs, ignore_index=True)
df_stock_price

'M' is deprecated and will be removed in a future version, please use 'ME' instead.
'M' is deprecated and will be removed in a future version, please use 'ME' instead.
'M' is deprecated and will be removed in a future version, please use 'ME' instead.
'M' is deprecated and will be removed in a future version, please use 'ME' instead.
'M' is deprecated and will be removed in a future version, please use 'ME' instead.
'M' is deprecated and will be removed in a future version, please use 'ME' instead.
'M' is deprecated and will be removed in a future version, please use 'ME' instead.
'M' is deprecated and will be removed in a future version, please use 'ME' instead.
'M' is deprecated and will be removed in a future version, please use 'ME' instead.
'M' is deprecated and will be removed in a future version, please use 'ME' instead.
'M' is deprecated and will be removed in a future version, please use 'ME' instead.
'M' is deprecated and will be removed in a future version, please use 'ME' i

,time,open,high,low,close,volume,stock_code
0,2015-01-31,6.69,6.77,6.02,6.02,2803900,HSG
1,2015-02-28,6.14,6.21,5.38,5.45,3432560,HSG
2,2015-03-31,5.43,5.59,4.67,4.78,7807130,HSG
3,2015-04-30,4.78,5.50,4.49,5.35,5360100,HSG
4,2015-05-31,5.38,5.38,4.94,5.18,2956820,HSG
...,...,...,...,...,...,...,...
11096,2024-08-31,13.00,13.30,11.90,12.90,49902800,ORS
11097,2024-09-30,12.60,13.30,12.00,13.20,33767500,ORS
11098,2024-10-31,13.30,15.05,13.20,14.25,195394800,ORS
11099,2024-11-30,14.15,14.30,12.45,13.90,107239600,ORS


In [4]:
df_stock_price['year'] = df_stock_price['time'].dt.year
df_stock_price['month'] = df_stock_price['time'].dt.month

df_price_quarter = df_stock_price[df_stock_price['month'].isin([3,6,9,12])]
df_price_quarter['quarter'] = df_price_quarter['month'] // 3

df_price_quarter_4 = df_price_quarter[df_price_quarter['quarter'] == 4]
df_price_quarter_4['quarter'] = 0

df_price_quarter = pd.concat([df_price_quarter, df_price_quarter_4], ignore_index=True)


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [6]:
df_price_quarter['open'] *= 1000
df_price_quarter['high'] *= 1000
df_price_quarter['low'] *= 1000
df_price_quarter['close'] *= 1000

df_price_quarter.to_parquet('../../data/stock_price_quarterly.parquet', index=False)

In [13]:
import pandas as pd

df = pd.read_parquet('../../data/financial_statement_v3.parquet')

In [14]:
df['stock_code'].unique()

array(['BID', 'CTG', 'VCB', 'ACB', 'MBB', 'HDB', 'TPB', 'VPB', 'STB',
       'TCB', 'SHB', 'VIB', 'LPB', 'SSI', 'MWG', 'VNM', 'HPG', 'VHM',
       'FPT', 'MSN', 'GAS', 'VRE', 'VJC', 'VIC', 'PLX', 'SAB', 'GVR',
       'BCM', 'DGC', 'Banking', 'Basic Resources', 'Chemicals',
       'Financial Services', 'Food and Beverages',
       'Information Technology', 'Oil and Gas', 'Real Estate', 'Retail',
       'Travel and Leisure', 'Utilities (Electricity, Water & Gas)'],
      dtype=object)

In [15]:
df.head()

,data,stock_code,quarter,year,date_added,segment,category_code
0,3253.384,BID,0.0,2010.0,2010-12-30,bank,BS_111
1,3628.604,BID,0.0,2011.0,2011-12-30,bank,BS_111
2,3295.068,BID,0.0,2012.0,2012-12-30,bank,BS_111
3,3862.664,BID,0.0,2013.0,2013-12-30,bank,BS_111
4,5393.484,BID,0.0,2014.0,2014-12-30,bank,BS_111
